## Querying PDF using **Langchain** with **Astra**

In [15]:
# !pip install -q cassio datasets langchain openai tiktoken

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ['GOOGLE_API_KEY'] = os.getenv("GOOGLE_API_KEY")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = "true"
os.environ['LANGCHAIN_PROJECT'] = "PDF Query"
os.environ['HF_TOKEN'] = os.getenv('HF_TOKEN')

In [3]:
from langchain_community.vectorstores.cassandra import Cassandra
from langchain_classic.indexes.vectorstore import VectorStoreIndexWrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings

from datasets import load_dataset

import cassio

c:\Users\singh\OneDrive\Desktop\Generative AI\GenAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# !pip install PyPDF2 

### Importing the PDF

In [5]:
from PyPDF2 import PdfReader

In [6]:
from typing_extensions import Concatenate

pdf = PdfReader('budget_speech.pdf')
raw_text = ''

for i, page in enumerate(pdf.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [7]:
raw_text

'GOVERNMENT OF INDIA\nBUDGET 2026-2027\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2026 \nCONTENTS  \n \nPage No.  \nIntroduction  1 \n                                                          PART - A                                          \nYuva Shakti and 3 kartavya  2 \nReform Express  3 \nFirst kartavya : to accelerate and sustain economic growth  3 \nSecond kartavya: fulfil aspirations and build capacity  10 \nThird kartavya:  Sabka Sath, Sabka Vikas  14  \n16th Finance Commission  18 \nFiscal Consolidation  18 \n \nPART – B \nDirect taxes  20 \nIndirect Taxes   26 \n \nAnnexure to Part -A 32 \nAnnexure to Part -B \nAmendments relating to Direct Taxes  33 \nAmendments relating to Indirect Taxes  50 \n \n \n   \n \nBudget 202 6-2027 \n \nSpeech of  \nNirmala Sitharaman  \nMinister of Finance  \nFebruary 1 , 202 6 \n \nHon’ble Speaker,  \nOn the sacred occasion of Magha Purnima and the birth \nanniversary of Guru Ravidas, I present the Budget for the year 2

### Initialize the Connection with Database

In [8]:
cassio.init(token= os.getenv('ASTRA_DB_APPLICATION_TOKEN'), database_id= os.getenv('ASTA_DB_ID'))

In [14]:
## Calling the LLm and Embedding Model
llm = ChatGoogleGenerativeAI(
    model= 'gemini-2.5-flash',
    google_api_key = os.getenv('GOOGLE_API_KEY')
)

embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-large-en-v1.5')

c:\Users\singh\OneDrive\Desktop\Generative AI\GenAI\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\singh\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 8061.89it/s]
BertModel LOAD REPORT fro

In [15]:
astra_vector_store = Cassandra(
    embedding= embeddings,
    table_name= 'qa_mini_demo',
    session=None, 
    keyspace=None
)

In [16]:
from langchain_text_splitters import CharacterTextSplitter
splitter = CharacterTextSplitter(
    separator= '\n',
    chunk_size= 800,
    chunk_overlap= 200,
    length_function = len
)

texts = splitter.split_text(raw_text)

In [18]:
texts[:50]

['GOVERNMENT OF INDIA\nBUDGET 2026-2027\nSPEECH\nOF\nNIRMALA SITHARAMAN\nMINISTER OF FINANCE\nFebruary 1,  2026 \nCONTENTS  \n \nPage No.  \nIntroduction  1 \n                                                          PART - A                                          \nYuva Shakti and 3 kartavya  2 \nReform Express  3 \nFirst kartavya : to accelerate and sustain economic growth  3 \nSecond kartavya: fulfil aspirations and build capacity  10 \nThird kartavya:  Sabka Sath, Sabka Vikas  14  \n16th Finance Commission  18 \nFiscal Consolidation  18 \n \nPART – B \nDirect taxes  20 \nIndirect Taxes   26 \n \nAnnexure to Part -A 32 \nAnnexure to Part -B \nAmendments relating to Direct Taxes  33 \nAmendments relating to Indirect Taxes  50 \n \n \n   \n \nBudget 202 6-2027 \n \nSpeech of  \nNirmala Sitharaman  \nMinister of Finance',
 'Annexure to Part -B \nAmendments relating to Direct Taxes  33 \nAmendments relating to Indirect Taxes  50 \n \n \n   \n \nBudget 202 6-2027 \n \nSpeech of  \nNirm

In [17]:
astra_vector_store.add_texts(texts)

print("Inserted %i Headlines." % len(texts))
astra_vector_index = VectorStoreIndexWrapper(vectorstore= astra_vector_store)

Inserted 171 Headlines.


### Asking Questions

In [19]:
first_question = True
while True:
    if first_question:
        query_text = input("\n Enter your Question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("\n What's your Next Question (or type 'quit' to exit): ").strip()
    
    if query_text.lower() == 'quit':
        break

    if query_text == "":
        continue

    first_question = False

    print("\nQUESTION : \"%s\"" % query_text)
    answer = astra_vector_index.query(query_text, llm=llm)
    print("ANSWER: \"%s\"" % answer)

    print("FIRST DOCUMENT BY RELEVANCE:")
    for doc, score in astra_vector_store.similarity_search_with_score(query_text, k=4):
        print("     [%0.4f]  \"%s ...\"" % (score, doc.page_content[:84]))    


QUESTION : "What is the Current GDP?"
ANSWER: "I'm sorry, but the provided context does not mention the current GDP. It only mentions debt-to-GDP ratio and fiscal deficit as a percentage of GDP for future estimates."
FIRST DOCUMENT BY RELEVANCE:
     [0.8137]  "92. Government has been delivering on our fiscal commitments 
consistently without c ..."
     [0.7916]  "payments.  
94. One of the main operational instruments for debt targeting is the 
f ..."
     [0.7823]  "Annexure to Part -B 
Amendments relating to Direct Taxes  33 
Amendments relating to ..."
     [0.7777]  "from every action of the Government, undertaking reforms to support  2  
 
employmen ..."

QUESTION : "City Economic Regions"
ANSWER: "Cities are recognized as India's engines of growth, innovation, and opportunities. The budget aims to amplify the potential of Tier II and Tier III cities, and even temple-towns, by mapping City Economic Regions (CER) based on their specific growth drivers.

For these CERs, an alloca